# 19 Paper Figures And Tables

Collect final publication-quality figures and summary tables for paper drafting.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
# Recreate a compact publication summary without depending on prior notebook execution order.
spot = 24000.0; K = 24000.0; T = 30/365; r = 0.065; sigma = 0.18; option_type = "put"
q = quantum_price_reconstruction(spot, K, T, r, sigma, option_type, n_qubits=8, x_width=float(config["quantum"]["x_width"]))
metrics = price_error_metrics(q["classical_curve"], q["price_curve"], q["S_grid"], K, option_type)
resources = resource_estimate_table(config["quantum"]["scaling_qubits"])
positions = load_portfolio_config()
valued, totals = value_portfolio(positions)
pnl = monte_carlo_portfolio_pnl(positions, n_scenarios=2000, horizon_days=1, seed=int(config["random_seed"]))
risk = var_expected_shortfall(pnl, config["stress"]["var_levels"])
summary = pd.DataFrame([
    {"metric": "Quantum price RMSE", "value": metrics["RMSE"]},
    {"metric": "Quantum max abs error", "value": metrics["MaxAbsError"]},
    {"metric": "Post-selection probability", "value": q["post_selection_probability"]},
    {"metric": "Portfolio theoretical value", "value": totals["total_theoretical_value"]},
    {"metric": "VaR 95", "value": risk["VaR_95"]},
    {"metric": "ES 95", "value": risk["ES_95"]},
])
save_table(summary, "19_publication_summary_metrics.csv")
plt.figure(figsize=(7, 4))
plt.plot(q["S_grid"], q["classical_curve"], label="Black-Scholes")
plt.plot(q["S_grid"], q["price_curve"], "--", label="Quantum reconstructed")
plt.title("Paper figure: reconstructed option curve")
plt.xlabel("Index level")
plt.ylabel("Put value")
plt.legend()
save_current_figure("19_paper_quantum_curve.png", dpi=200)
plt.figure(figsize=(7, 4))
plt.semilogy(resources["grid_qubits"], resources["complex128_bytes"] / 1e12, marker="o")
plt.title("Paper figure: memory scaling")
plt.xlabel("Grid qubits")
plt.ylabel("complex128 memory (TB)")
save_current_figure("19_paper_memory_scaling.png", dpi=200)
# Refresh paper results summary.
summary_block = summary.to_string(index=False)
(PAPER_DIR / "results_summary.md").write_text("# Results Summary\n\n```text\n" + summary_block + "\n```\n\nGenerated by notebook 19. Synthetic examples only; no quantum advantage claimed.\n", encoding="utf-8")
summary
